In [6]:
"""
Vorbereitung von Fahrzeugbeständen und Emissionsparametern

Dieses Script:
- lädt den finalen Fahrzeugbestand aus Results/policy_final_stock_all.csv
- lädt eine separate Emissionsparameter-Tabelle im breiten Format
- bereinigt und transformiert die Emissionsparameter ins lange Format
- ergänzt die Bestandstabelle um drivetrain_group_std und size_class
- prüft, ob alle relevanten Kombinationen aus Antrieb und Größenklasse
  passende Emissionsparameter besitzen

Es führt noch keine Emissionsberechnung durch.
"""

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. Zentrale Pfade und Einstellungen
# ============================================================

RESULTS_DIR = Path("Results")

STOCK_PATH = RESULTS_DIR / "policy_final_stock_all.csv"

# Diesen Pfad bitte an deine tatsächliche Datei anpassen
EMISSION_PARAMS_PATH = Path(
    r"C:\Users\annic\OneDrive\Dokumente\Master Technischer Umweltschutz\Master Thesis\Emission_Trajectory_Thesis\Fleet Model\LCA_Emissions_Python.csv"
) 

CSV_SEPARATOR = ";"

NA_VALUES = ["", "na", "NA", "nan", "NaN", "NAN"]


# ============================================================
# 2. Hilfsfunktionen
# ============================================================

def require_file(path: Path, description: str) -> None:
    """
    Prüft, ob eine Datei existiert. Falls nicht, wird eine verständliche
    Fehlermeldung ausgegeben.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{description} wurde nicht gefunden:\n"
            f"  {path.resolve()}\n\n"
            f"Bitte prüfe den Dateipfad."
        )


def to_numeric_series(series: pd.Series) -> pd.Series:
    """
    Wandelt eine Spalte robust in numerische Werte um.
    Unterstützt deutsche Dezimalkommas, z. B. '5,00'.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace(NA_VALUES, pd.NA)
        .pipe(pd.to_numeric, errors="coerce")
    )


def standardize_size_class(value: str) -> str:
    """
    Standardisiert Größenklassen auf Small, Medium, Large.
    """
    if pd.isna(value):
        return np.nan

    value_clean = str(value).strip().lower()

    mapping = {
        "small": "Small",
        "medium": "Medium",
        "large": "Large",
    }

    return mapping.get(value_clean, value)


def map_drivetrain_to_emission_group(drivetrain: str) -> str:
    """
    Ordnet die Antriebsarten aus dem Flottenmodell den Emissionsgruppen zu.

    Nicht eindeutig zuordenbare Antriebe werden als 'Other' markiert.
    Die Regeln sind bewusst robust gegen unterschiedliche Schreibweisen.
    """
    if pd.isna(drivetrain):
        return "Other"

    text = str(drivetrain).strip().lower()

    # Elektrofahrzeuge
    if (
        "elektro" in text
        or "electric" in text
        or "bev" in text
        or "battery" in text
    ):
        return "BEV"

    # Diesel
    if "diesel" in text:
        return "ICE_Diesel"

    # Benzin / Petrol / Otto
    if (
        "benzin" in text
        or "petrol" in text
        or "gasoline" in text
        or "otto" in text
    ):
        return "ICE_Petrol"

    return "Other"


def map_segment_to_size_class(segment: str) -> str:
    """
    Ordnet die Segmente aus dem Flottenmodell den Größenklassen zu.
    """
    if pd.isna(segment):
        return np.nan

    segment_clean = str(segment).strip()

    segment_to_size = {
        "Minis": "Small",
        "Kleinwagen": "Small",

        "Kompaktklasse": "Medium",
        "Mittelklasse": "Medium",
        "Mini-Vans": "Medium",
        "Sonstige": "Medium",

        "Obere Mittelklasse": "Large",
        "Oberklasse": "Large",
        "SUVs": "Large",
        "Geländewagen": "Large",
        "Sportwagen": "Large",
        "Großraum-Vans": "Large",
        "Utilities": "Large",
        "Wohnmobile": "Large",
    }

    return segment_to_size.get(segment_clean, np.nan)


# ============================================================
# 3. Fahrzeugbestand laden und vorbereiten
# ============================================================

def load_stock_data(stock_path: Path) -> pd.DataFrame:
    """
    Lädt die finale Bestandstabelle aus dem Flottenmodell.
    """
    require_file(stock_path, "Die finale Bestandstabelle")

    stock = pd.read_csv(
        stock_path,
        sep=CSV_SEPARATOR,
        na_values=NA_VALUES,
        keep_default_na=True,
        encoding="utf-8-sig",
    )

    required_columns = [
        "Berichtsjahr",
        "Segment",
        "Antriebsart",
        "Jahr der Erstzulassung",
        "finaler_bestand",
        "Szenario",
        "Jahr",
    ]

    missing_columns = [col for col in required_columns if col not in stock.columns]
    if missing_columns:
        raise ValueError(
            "Die Bestandstabelle enthält nicht alle erwarteten Spalten.\n"
            f"Fehlende Spalten: {missing_columns}\n"
            f"Gefundene Spalten: {list(stock.columns)}"
        )

    # Jahresangaben und Bestände numerisch einlesen bzw. konvertieren
    numeric_columns = [
        "Berichtsjahr",
        "Jahr der Erstzulassung",
        "finaler_bestand",
        "Jahr",
    ]

    for col in numeric_columns:
        stock[col] = to_numeric_series(stock[col])

    # Ganzzahlige Jahresangaben nullable speichern
    for col in ["Berichtsjahr", "Jahr der Erstzulassung", "Jahr"]:
        stock[col] = stock[col].astype("Int64")

    return stock


# ============================================================
# 4. Emissionsparameter laden, bereinigen und transformieren
# ============================================================

def load_and_prepare_emission_params(params_path: Path) -> pd.DataFrame:
    """
    Lädt die Emissionsparameter-Tabelle im breiten Format und wandelt sie
    in ein langes, modellkompatibles Format um.
    """
    require_file(params_path, "Die Emissionsparameter-Datei")

    params_wide = pd.read_csv(
        params_path,
        sep=CSV_SEPARATOR,
        na_values=NA_VALUES,
        keep_default_na=True,
        encoding="utf-8-sig",
        dtype="string",
    )

    required_text_columns = ["Parameter", "Explanation_parameter"]

    missing_text_columns = [
        col for col in required_text_columns if col not in params_wide.columns
    ]
    if missing_text_columns:
        raise ValueError(
            "Die Emissionsparameter-Datei enthält nicht alle erwarteten Textspalten.\n"
            f"Fehlende Spalten: {missing_text_columns}\n"
            f"Gefundene Spalten: {list(params_wide.columns)}"
        )

    # Technologie-Spalten sind alle Spalten außer Parameter und Explanation_parameter
    tech_columns = [
        col for col in params_wide.columns
        if col not in required_text_columns
    ]

    if not tech_columns:
        raise ValueError(
            "In der Emissionsparameter-Datei wurden keine Technologie-Spalten gefunden."
        )

    # Emissionswerte numerisch konvertieren
    for col in tech_columns:
        params_wide[col] = to_numeric_series(params_wide[col])

    # Breites Format in langes Format umwandeln
    params_long = params_wide.melt(
        id_vars=["Parameter", "Explanation_parameter"],
        value_vars=tech_columns,
        var_name="tech_size",
        value_name="value",
    )

    # Leere Parameterwerte optional entfernen
    params_long = params_long.dropna(subset=["Parameter", "tech_size"])

    # tech_size zerlegen:
    # BEV_small -> BEV + small
    # ICEVs-Petrol_medium -> ICEVs-Petrol + medium
    # ICEVs-Diesel_large -> ICEVs-Diesel + large
    split_cols = params_long["tech_size"].str.rsplit("_", n=1, expand=True)

    if split_cols.shape[1] != 2:
        invalid_values = params_long.loc[
            split_cols.shape[1] != 2 if isinstance(split_cols.shape[1], pd.Series) else [],
            "tech_size",
        ].unique()

        raise ValueError(
            "Mindestens eine tech_size-Spalte konnte nicht nach dem Muster "
            "'drivetrain_size' zerlegt werden.\n"
            f"Beispiele für erwartete Werte: BEV_small, ICEVs-Petrol_medium\n"
            f"Problematische Werte: {invalid_values}"
        )

    params_long["drivetrain_group"] = split_cols[0]
    params_long["size_class"] = split_cols[1].apply(standardize_size_class)

    # Antriebsgruppen standardisieren
    drivetrain_standardization = {
        "BEV": "BEV",
        "ICEVs-Petrol": "ICE_Petrol",
        "ICEVs-Diesel": "ICE_Diesel",
    }

    params_long["drivetrain_group_std"] = (
        params_long["drivetrain_group"]
        .map(drivetrain_standardization)
    )

    # Prüfen, ob unerwartete Technologiegruppen vorhanden sind
    unknown_drivetrain_groups = (
        params_long.loc[
            params_long["drivetrain_group_std"].isna(),
            "drivetrain_group",
        ]
        .dropna()
        .unique()
    )

    if len(unknown_drivetrain_groups) > 0:
        print("\nWARNUNG: Einige Emissions-Antriebsgruppen konnten nicht standardisiert werden:")
        print(sorted(unknown_drivetrain_groups))

    # Spalten in gewünschter Reihenfolge
    params_long = params_long[
        [
            "Parameter",
            "Explanation_parameter",
            "tech_size",
            "value",
            "drivetrain_group",
            "size_class",
            "drivetrain_group_std",
        ]
    ]

    return params_long


# ============================================================
# 5. Bestandstabelle um Emissionsgruppen und Größenklassen ergänzen
# ============================================================

def enrich_stock_with_emission_keys(stock: pd.DataFrame) -> pd.DataFrame:
    """
    Ergänzt die Bestandstabelle um:
    - drivetrain_group_std
    - size_class
    """
    stock = stock.copy()

    stock["drivetrain_group_std"] = stock["Antriebsart"].apply(
        map_drivetrain_to_emission_group
    )

    stock["size_class"] = stock["Segment"].apply(
        map_segment_to_size_class
    )

    return stock


# ============================================================
# 6. Prüfungen und Diagnose-Ausgaben
# ============================================================

def print_diagnostics(
    stock_prepared: pd.DataFrame,
    emission_params_long: pd.DataFrame,
) -> None:
    """
    Gibt zentrale Prüf- und Diagnoseinformationen aus.
    """
    print("\n" + "=" * 80)
    print("DIAGNOSE DER VORBEREITETEN DATEN")
    print("=" * 80)

    # Szenarien
    scenarios = (
        stock_prepared["Szenario"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print("\nEnthaltene Szenarien:")
    print(scenarios)

    # Jahre
    years = (
        stock_prepared["Jahr"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print("\nEnthaltene Jahre:")
    if years:
        print(f"{min(years)} bis {max(years)}")
        print(years)
    else:
        print("Keine Jahre gefunden.")

    # Nicht eindeutig zuordenbare Antriebsarten
    unmapped_drivetrains = (
        stock_prepared.loc[
            stock_prepared["drivetrain_group_std"] == "Other",
            "Antriebsart",
        ]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print("\nNicht eindeutig zuordenbare Antriebsarten:")
    if unmapped_drivetrains:
        print(unmapped_drivetrains)
    else:
        print("Keine. Alle Antriebsarten konnten zugeordnet werden.")

    # Segmentnamen, die geprüft werden sollten
    segments_to_check = (
        stock_prepared.loc[
            stock_prepared["size_class"].isna(),
            "Segment",
        ]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print("\nSegmentnamen, die eventuell geprüft werden sollten:")
    if segments_to_check:
        print(segments_to_check)
    else:
        print("Keine. Alle Segmente konnten einer Größenklasse zugeordnet werden.")

    # Kombinationen aus Antrieb und Größenklasse im Bestand
    stock_combinations = (
        stock_prepared.loc[
            (stock_prepared["drivetrain_group_std"] != "Other")
            & (stock_prepared["size_class"].notna()),
            ["drivetrain_group_std", "size_class"]
        ]
        .drop_duplicates()
        .sort_values(["drivetrain_group_std", "size_class"])
    )

    # Kombinationen in Emissionsparametern
    emission_combinations = (
        emission_params_long.loc[
            emission_params_long["drivetrain_group_std"].notna()
            & emission_params_long["size_class"].notna(),
            ["drivetrain_group_std", "size_class"]
        ]
        .drop_duplicates()
        .sort_values(["drivetrain_group_std", "size_class"])
    )

    missing_combinations = stock_combinations.merge(
        emission_combinations,
        on=["drivetrain_group_std", "size_class"],
        how="left",
        indicator=True,
    )

    missing_combinations = missing_combinations.loc[
        missing_combinations["_merge"] == "left_only",
        ["drivetrain_group_std", "size_class"],
    ]

    print("\nKombinationen aus Antrieb und Größenklasse ohne passende Emissionsparameter:")
    if not missing_combinations.empty:
        print(missing_combinations.to_string(index=False))
    else:
        print("Keine. Für alle relevanten Kombinationen sind Emissionsparameter vorhanden.")

    print("\nErste Zeilen der vorbereiteten Emissionsparameter-Tabelle:")
    print(emission_params_long.head(20).to_string(index=False))

    print("\nDimensionen:")
    print(f"Bestandstabelle vorbereitet: {stock_prepared.shape[0]:,} Zeilen, {stock_prepared.shape[1]} Spalten")
    print(f"Emissionsparameter lang:     {emission_params_long.shape[0]:,} Zeilen, {emission_params_long.shape[1]} Spalten")


# ============================================================
# 7. Hauptprogramm
# ============================================================

def main() -> None:
    print("Lade finale Bestandstabelle...")
    stock = load_stock_data(STOCK_PATH)

    print("Lade und bereite Emissionsparameter vor...")
    emission_params_long = load_and_prepare_emission_params(EMISSION_PARAMS_PATH)

    print("Ergänze Bestandstabelle um Emissions-Zuordnungsschlüssel...")
    stock_prepared = enrich_stock_with_emission_keys(stock)

    print_diagnostics(
        stock_prepared=stock_prepared,
        emission_params_long=emission_params_long,
    )

    # Optional: vorbereitete Tabellen speichern
    # Diese Dateien können im nächsten Schritt direkt für den Merge verwendet werden.
    prepared_stock_path = RESULTS_DIR / "policy_final_stock_all_prepared_for_emissions.csv"
    prepared_params_path = RESULTS_DIR / "emission_parameters_long_prepared.csv"

    stock_prepared.to_csv(
        prepared_stock_path,
        sep=CSV_SEPARATOR,
        index=False,
        encoding="utf-8-sig",
    )

    emission_params_long.to_csv(
        prepared_params_path,
        sep=CSV_SEPARATOR,
        index=False,
        encoding="utf-8-sig",
    )

    print("\nVorbereitete Dateien gespeichert:")
    print(f"- {prepared_stock_path}")
    print(f"- {prepared_params_path}")

    print("\nFertig. Es wurde noch keine Emissionsberechnung durchgeführt.")


if __name__ == "__main__":
    main()

Lade finale Bestandstabelle...


Lade und bereite Emissionsparameter vor...
Ergänze Bestandstabelle um Emissions-Zuordnungsschlüssel...

DIAGNOSE DER VORBEREITETEN DATEN

Enthaltene Szenarien:
['abwrackpraemie', 'baseline', 'diesel_fahrverbot', 'verbrenneraus']

Enthaltene Jahre:
2025 bis 2050
[2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050]

Nicht eindeutig zuordenbare Antriebsarten:
['Erdgas (CNG) (einschl. bivalent)', 'Flüssiggas (LPG) (einschl. bivalent)', 'Hybrid', 'Sonstige']

Segmentnamen, die eventuell geprüft werden sollten:
Keine. Alle Segmente konnten einer Größenklasse zugeordnet werden.

Kombinationen aus Antrieb und Größenklasse ohne passende Emissionsparameter:
Keine. Für alle relevanten Kombinationen sind Emissionsparameter vorhanden.

Erste Zeilen der vorbereiteten Emissionsparameter-Tabelle:
            Parameter                                                                          Explanati